# Pseudobulk model output check

**Environment:** `clamp-analyses`

Inspects all model outputs in `output/01_model_building/05_pseudobulk/`. For each dataset × method:
- Reports k from `k.csv`
- Checks B matrix (LV × samples) and Z matrix (genes × LV) dimensions
- Shows `head()` of B and Z
- Flags missing files

## Libraries

In [ ]:
library(data.table)
library(here)

## Configuration

In [ ]:
OUT_ROOT <- here("output", "01_model_building", "05_pseudobulk")

DATASETS <- c(
  "Brain_Mathys2023", "Brain_Xiong2023", "BRCA_Bassez2021",
  "CRC_Pelka2021",    "PBMC_1k1k",       "PBMC_Perez2022"
)

# method name -> list(B = relative path, Z = relative path or NA)
METHODS <- list(
  CLAMPbase       = list(B = "CLAMPbase/B.csv",          Z = "CLAMPbase/Z.csv"),
  CLAMPfull       = list(B = "CLAMPfull/B.csv",          Z = "CLAMPfull/Z.csv"),
  PLIER           = list(B = "PLIER/B.csv",              Z = "PLIER/Z.csv"),
  PCA             = list(B = "PCA/B.csv",                Z = "PCA/Z.csv"),
  NMF             = list(B = "NMF/B.csv",                Z = "NMF/Z.csv"),
  ICA             = list(B = "ICA/B.csv",                Z = "ICA/Z.csv"),
  flashier        = list(B = "flashier/B.csv",           Z = "flashier/Z.csv"),
  MOFA_FLEX_PRIOR = list(B = "MOFA_FLEX_PRIOR/B_matrix.csv", Z = NA),
  GSS             = list(B = "GSS/B.csv",               Z = NA)
)

## Helpers

In [ ]:
check_matrix <- function(path, label) {
  if (!file.exists(path)) {
    cat(sprintf("  [MISSING] %s: %s\n", label, basename(path)))
    return(invisible(NULL))
  }
  dt  <- fread(path, nrows = 5)
  full <- fread(path)
  n_rows <- nrow(full)
  n_cols <- ncol(full) - 1L  # subtract row-name column
  cat(sprintf("  [OK] %s: %d x %d  (head below)\n", label, n_rows, n_cols))
  # Print head: first 5 rows, first 6 columns (including row names)
  cols_show <- min(ncol(dt), 6L)
  print(dt[, seq_len(cols_show), with = FALSE])
  cat("\n")
  invisible(full)
}

## Inspect all outputs

In [ ]:
issues <- character(0)

for (ds in DATASETS) {
  ds_dir  <- file.path(OUT_ROOT, ds)
  k_path  <- file.path(ds_dir, "k.csv")

  cat("\n", strrep("=", 60), "\n", sep = "")
  cat(ds, "\n")
  cat(strrep("=", 60), "\n")

  if (!file.exists(k_path)) {
    cat("  [MISSING] k.csv\n")
    issues <- c(issues, paste(ds, "k.csv missing"))
    next
  }
  k <- as.integer(fread(k_path)$k[1])
  cat(sprintf("  k = %d\n", k))

  for (meth in names(METHODS)) {
    cfg <- METHODS[[meth]]
    cat(sprintf("\n  --- %s ---\n", meth))

    b_path <- file.path(ds_dir, cfg$B)
    B <- check_matrix(b_path, "B (LV x samples)")
    if (is.null(B)) issues <- c(issues, paste(ds, meth, "B missing"))

    if (!is.na(cfg$Z)) {
      z_path <- file.path(ds_dir, cfg$Z)
      Z <- check_matrix(z_path, "Z (genes x LV)")
      if (is.null(Z)) issues <- c(issues, paste(ds, meth, "Z missing"))
    }
  }
}

cat("\n", strrep("=", 60), "\n", sep = "")
cat("SUMMARY\n")
cat(strrep("=", 60), "\n")
if (length(issues) == 0) {
  cat("All checked outputs present.\n")
} else {
  cat(sprintf("%d issue(s):\n", length(issues)))
  for (iss in issues) cat(" -", iss, "\n")
}